In [ ]:
!pip install -q transformers datasets evaluate gradio

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

# Loaded the same Turkish dataset
dataset = load_dataset("sepidmnorozy/Turkish_sentiment")

# Loaded BERTurk tokenizer
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=128)

tokenized = dataset.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

print("BERTurk loaded and ready!")

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"]
    }

args = TrainingArguments(
    output_dir="berturk-sentiment",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate(tokenized["test"])
print(f"BERTurk Accuracy: {results['eval_accuracy']:.4f}")
print(f"BERTurk F1:       {results['eval_f1']:.4f}")
print()
print("=== FULL COMPARISON ===")
print(f"English DistilBERT zero-shot: 0.4171")
print(f"mBERT fine-tuned:             0.8436")
print(f"BERTurk fine-tuned:           {results['eval_accuracy']:.4f}")
print(f"BERTurk gain over mBERT:      +{results['eval_accuracy']-0.8436:.4f}")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
trainer.push_to_hub("berturk-sentiment")